<a href="https://colab.research.google.com/github/kmabro/Automated-Certificate-Generation-MATLAB/blob/main/MT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# IndicTrans2


In [1]:
%%capture
!git clone https://github.com/AI4Bharat/IndicTrans2.git

In [2]:
%%capture
%cd /content/IndicTrans2/huggingface_interface

In [3]:
%%capture
!python3 -m pip install nltk sacremoses pandas regex mock transformers==4.53.2 mosestokenizer
!python3 -c "import nltk; nltk.download('punkt')"
!python3 -m pip install bitsandbytes scipy accelerate datasets
!python3 -m pip install sentencepiece

!git clone https://github.com/VarunGumma/IndicTransToolkit.git
%cd IndicTransToolkit
!python3 -m pip install --editable ./
%cd ..

In [4]:
%cd /content/IndicTrans2/huggingface_interface/IndicTransToolkit
!python -m pip install --upgrade --force-reinstall --no-deps .

/content/IndicTrans2/huggingface_interface/IndicTransToolkit
Processing ./.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for indictranstoolkit: filename=indictranstoolkit-1.1.1-cp313-cp313-linux_x86_64.whl size=490019 sha256=3b7835584d7f31a1e240a89ca227b9560b94d9a4f4c242d3dbf5f5402bc02147
  Stored in directory: /tmp/pip-ephem-wheel-cache-v3gnw1kj/wheels/be/29/4e/ca5b0121ffb9ad77cc585f3552e1184351c1fff878b73285ab
Successfully built indictranstoolkit
  Attempting uninstall: indictranstoolkit
    Found existing installation: indictranstoolkit 1.1.1
    Uninstalling indictranstoolkit-1.1.1:
      Successfully uninstalled indictranstoolkit-1.1.1


In [6]:
from IndicTransToolkit import IndicProcessor

ip = IndicProcessor(inference=True)

print("IndicProcessor imported successfully!")

IndicProcessor imported successfully!


In [7]:

import torch
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)

def initialize_model_and_tokenizer(ckpt_dir, quantization=None):
    if quantization == "4-bit":
        qconfig = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.bfloat16,
        )
    elif quantization == "8-bit":
        qconfig = BitsAndBytesConfig(
            load_in_8bit=True,
        )
    else:
        qconfig = None

    tokenizer = AutoTokenizer.from_pretrained(
        ckpt_dir,
        trust_remote_code=True,
    )

    model = AutoModelForSeq2SeqLM.from_pretrained(
        ckpt_dir,
        trust_remote_code=True,
        low_cpu_mem_usage=True,
        quantization_config=qconfig,
    )

    if qconfig is None:
        model = model.to(DEVICE)

        if DEVICE == "cuda":
            model.half()

    model.eval()

    return tokenizer, model


def batch_translate(
    input_sentences,
    src_lang,
    tgt_lang,
    model,
    tokenizer,
    ip,
):
    if not input_sentences:
        return []

    translations = []

    for i in range(0, len(input_sentences), BATCH_SIZE):
        batch = input_sentences[i:i + BATCH_SIZE]

        # Preprocess the input sentences
        batch = ip.preprocess_batch(
            batch,
            src_lang=src_lang,
            tgt_lang=tgt_lang,
        )

        # Tokenize and move inputs to the selected device
        inputs = tokenizer(
            batch,
            truncation=True,
            padding="longest",
            return_tensors="pt",
            return_attention_mask=True,
        ).to(DEVICE)

        # Generate translations
        with torch.no_grad():
            generated_tokens = model.generate(
                **inputs,
                use_cache=True,
                min_length=0,
                max_length=256,
                num_beams=5,
                num_return_sequences=1,
            )

        # Decode model output
        outputs = tokenizer.batch_decode(
            generated_tokens.detach().cpu().tolist(),
            skip_special_tokens=True,
            clean_up_tokenization_spaces=True,
        )

        # Postprocess translated text
        translations.extend(
            ip.postprocess_batch(outputs, lang=tgt_lang)
        )

        del inputs, generated_tokens

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    return translations


print("Translation functions defined successfully!")


Translation functions defined successfully!


In [15]:
from huggingface_hub import notebook_login
notebook_login()

In [17]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_NAME = "ai4bharat/indictrans2-en-indic-1B"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", DEVICE)
print("Loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

print("Loading translation model...")

model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    low_cpu_mem_usage=True
)

model = model.to(DEVICE)

if DEVICE == "cuda":
    model.half()

model.eval()

print("\nModel and tokenizer loaded successfully!")
print("Tokenizer available:", tokenizer is not None)
print("Model available:", model is not None)
print("Device:", DEVICE)

Device: cuda
Loading tokenizer...


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenization_indictrans.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/ai4bharat/indictrans2-en-indic-1B:
- tokenization_indictrans.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


dict.SRC.json: 0.00B [00:00, ?B/s]

...

modeling_indictrans.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/ai4bharat/indictrans2-en-indic-1B:
- modeling_indictrans.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors:   0%|          | 0.00/4.46G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/163 [00:00<?, ?B/s]


Model and tokenizer loaded successfully!
Tokenizer available: True
Model available: True
Device: cuda


### English to Sindhi Example


In [19]:

import torch
from IndicTransToolkit import IndicProcessor

src_lang = "eng_Latn"
tgt_lang = "snd_Arab"

ip = IndicProcessor(inference=True)

input_sentences = [
    "Hello, how are you?",
    "My name is Khan.",
    "I live in Sindh, Pakistan.",
    "I am a computer systems engineering student.",
    "Artificial intelligence is changing the world.",
    "I want to translate English videos into Sindhi."
]

# Prepare English sentences for IndicTrans2
batch = ip.preprocess_batch(
    input_sentences,
    src_lang=src_lang,
    tgt_lang=tgt_lang
)

inputs = tokenizer(
    batch,
    truncation=True,
    padding="longest",
    return_tensors="pt",
    return_attention_mask=True
).to(DEVICE)

# Generate Sindhi translations
with torch.no_grad():
    generated_tokens = model.generate(
        **inputs,
        use_cache=True,
        min_length=0,
        max_length=256,
        num_beams=5,
        num_return_sequences=1
    )

outputs = tokenizer.batch_decode(
    generated_tokens.detach().cpu().tolist(),
    skip_special_tokens=True,
    clean_up_tokenization_spaces=True
)

translations = ip.postprocess_batch(outputs, lang=tgt_lang)

for english, sindhi in zip(input_sentences, translations):
    print("English:", english)
    print("Sindhi: ", sindhi)
    print("-" * 60)


English: Hello, how are you?
Sindhi:  ہیلو، ڪیئن آہیو؟
------------------------------------------------------------
English: My name is Khan.
Sindhi:  منھنجو نالو خان آھی۔
------------------------------------------------------------
English: I live in Sindh, Pakistan.
Sindhi:  مان سنڌ پاڪستان ۾ رہان ٿو۔
------------------------------------------------------------
English: I am a computer systems engineering student.
Sindhi:  مان ڪمپیوٽر سسٽم انجنئیرنگ جو شاگرد آہیان۔
------------------------------------------------------------
English: Artificial intelligence is changing the world.
Sindhi:  آرٽیفیشل انٽیلیجنس دنیا کی تبدیل ڪری رھیو آھی
------------------------------------------------------------
English: I want to translate English videos into Sindhi.
Sindhi:  مان انگریزی ویڊیوز جو سنڌی ۾ ترجمو ڪرڻ گھران ٿو۔
------------------------------------------------------------


In [20]:
import torch
from IndicTransToolkit import IndicProcessor

src_lang = "eng_Latn"
tgt_lang = "snd_Arab"

ip = IndicProcessor(inference=True)

input_sentences = [
    "Although artificial intelligence has made remarkable progress, translating spoken English into Sindhi remains challenging because of linguistic differences and limited training data.",
    "The automatic speech recognition system extracts timestamps and transcribes spoken words before the machine translation model converts the English transcript into Sindhi.",
    "When a speaker talks quickly, uses unfamiliar terminology, or changes their accent, the translation system must preserve the original meaning without introducing errors or omitting important information.",
    "The researchers decided to think outside the box because conventional translation methods were unable to capture the intended meaning of several English expressions.",
    "If the translated sentence is significantly longer than the original English sentence, the text-to-speech system may struggle to synchronize the Sindhi audio with the video.",
    "The engineer saw the man with the telescope while he was observing the machine near the laboratory.",
    "The model should not translate technical terms word for word when doing so would distort their meaning in the target language.",
    "Our system aims to generate natural-sounding Sindhi dubbing while preserving semantic accuracy, sentence timing, and the overall coherence of the original English dialogue.",
    "Despite the availability of multilingual transformer architectures, low-resource languages often suffer from inadequate parallel corpora, inconsistent orthography, and limited evaluation benchmarks.",
    "Had the original speaker paused more frequently between sentences, the dubbing system would have had greater flexibility to align the synthesized Sindhi speech with the available audio segments."
]

# Preprocess English text
batch = ip.preprocess_batch(
    input_sentences,
    src_lang=src_lang,
    tgt_lang=tgt_lang
)

inputs = tokenizer(
    batch,
    truncation=True,
    padding="longest",
    return_tensors="pt",
    return_attention_mask=True
).to(DEVICE)

# Translate into Sindhi
with torch.no_grad():
    generated_tokens = model.generate(
        **inputs,
        use_cache=True,
        min_length=0,
        max_length=256,
        num_beams=5,
        num_return_sequences=1
    )

# Decode and postprocess translations
outputs = tokenizer.batch_decode(
    generated_tokens.detach().cpu().tolist(),
    skip_special_tokens=True,
    clean_up_tokenization_spaces=True
)

translations = ip.postprocess_batch(outputs, lang=tgt_lang)

# Print results
for i, (english, sindhi) in enumerate(
    zip(input_sentences, translations), start=1
):
    print(f"\n{'=' * 80}")
    print(f"Sentence {i}")
    print(f"English: {english}")
    print(f"Sindhi:  {sindhi}")

print(f"\n{'=' * 80}")
print(f"Translation completed: {len(translations)} sentences")


Sentence 1
English: Although artificial intelligence has made remarkable progress, translating spoken English into Sindhi remains challenging because of linguistic differences and limited training data.
Sindhi:  جیتوڻیڪ مصنوعی ذہانت قابل ذڪر ترقی ڪئی آھی، بولیل انگریزی جو سنڌی ۾ ترجمو ڪرڻ مشڪل آھی ڇاڪاڻ تہ لسانی فرق ۽ محدود تربیتی ڊیٽا جی ڪری

Sentence 2
English: The automatic speech recognition system extracts timestamps and transcribes spoken words before the machine translation model converts the English transcript into Sindhi.
Sindhi:  مشینی ترجمو ماڊل انگریزی ٽرانسڪرپٽ کی سنڌی ۾ تبدیل ڪرڻ کان اڳ ۾ خودکار تقریر جی سڃاڻپ وارو نظام ٽائیم اسٽامپ ڪ extract ی ٿو ۽ بولیل لفظن کی نقل ڪری ٿو

Sentence 3
English: When a speaker talks quickly, uses unfamiliar terminology, or changes their accent, the translation system must preserve the original meaning without introducing errors or omitting important information.
Sindhi:  جڏھن ڪو تقریر ڪندڙ جلدی ڳالھائیندو آھی، اڻ واقف اصطلاحات استعمال ڪند